# AI-Based Phishing Website Detection System Using Machine Learning

**IBM SkillsBuild Cyber Security BootCamp 2026 — BharatCares**

**Author:** Rampravesh

---

## Table of Contents
1. [Project Overview](#1-project-overview)
2. [Library Imports](#2-library-imports)
3. [Dataset Loading](#3-dataset-loading)
4. [Exploratory Data Analysis (EDA)](#4-exploratory-data-analysis)
5. [Feature Engineering](#5-feature-engineering)
6. [Data Preprocessing](#6-data-preprocessing)
7. [Model Training](#7-model-training)
8. [Model Evaluation](#8-model-evaluation)
9. [Best Model Selection & Saving](#9-best-model-selection--saving)
10. [Live URL Prediction Demo](#10-live-url-prediction-demo)
11. [Summary](#11-summary)

## 1. Project Overview

Phishing websites impersonate legitimate sites to steal credentials, financial data,
and personal information. This project builds a Machine Learning pipeline that:

- Extracts **16 structural URL features** (length, dots, hyphens, HTTPS flag, IP
  address usage, suspicious TLD, phishing keywords, etc.)
- Trains **three classifiers** — Logistic Regression, Decision Tree, Random Forest
- Evaluates models using accuracy, precision, recall, F1-score, AUC-ROC, and
  cross-validation
- Selects and saves the best model for deployment in a Streamlit dashboard

### Dataset
**UCI Phishing Websites Dataset** (preferred)
- Source: https://archive.ics.uci.edu/dataset/327/phishing+websites
- 11,055 instances, 30 features, binary label (phishing / legitimate)
- Licence: CC BY 4.0

If the UCI dataset is not found in `dataset/phishing.arff`, a synthetic dataset
is generated based on published URL-pattern statistics.

## 2. Library Imports

In [ ]:
# Standard library
import os
import re
import sys
import json
import time
import urllib.parse
import warnings
warnings.filterwarnings('ignore')

# Data science
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

# Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
import joblib

print('All libraries imported successfully ✅')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')
import sklearn; print(f'Sklearn: {sklearn.__version__}')

## 3. Dataset Loading

The cell below checks for the UCI ARFF file first, then a CSV, and finally
falls back to a synthetic dataset.

In [ ]:
RANDOM_STATE = 42

# ---- UCI ARFF loader ----
def load_uci_arff(path):
    from scipy.io import arff
    data, meta = arff.loadarff(path)
    df = pd.DataFrame(data)
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].str.decode('utf-8')
    df['label'] = df['Result'].apply(lambda x: 0 if str(x).strip() == '-1' else 1)
    return df, 'UCI Phishing Websites Dataset (archive.ics.uci.edu/dataset/327)'

# ---- Synthetic dataset ----
def build_synthetic_dataset(n_samples=6000):
    rng = np.random.default_rng(RANDOM_STATE)
    n_each = n_samples // 2

    phish = {
        'url_length':           rng.integers(60, 200, n_each),
        'hostname_length':      rng.integers(20, 80, n_each),
        'path_length':          rng.integers(20, 120, n_each),
        'num_dots':             rng.integers(3, 10, n_each),
        'num_hyphens':          rng.integers(1, 8, n_each),
        'num_at':               rng.integers(0, 3, n_each),
        'num_question_marks':   rng.integers(0, 4, n_each),
        'num_equal':            rng.integers(0, 5, n_each),
        'num_slashes':          rng.integers(4, 12, n_each),
        'num_percent':          rng.integers(0, 6, n_each),
        'has_https':            rng.integers(0, 2, n_each),
        'has_ip':               rng.integers(0, 2, n_each),
        'has_suspicious_tld':   rng.integers(0, 2, n_each),
        'num_subdomains':       rng.integers(2, 6, n_each),
        'has_phishing_keyword': rng.integers(0, 2, n_each),
        'double_slash_in_path': rng.integers(0, 2, n_each),
        'label':                np.zeros(n_each, dtype=int),
    }
    legit = {
        'url_length':           rng.integers(15, 80, n_each),
        'hostname_length':      rng.integers(5, 30, n_each),
        'path_length':          rng.integers(0, 50, n_each),
        'num_dots':             rng.integers(1, 4, n_each),
        'num_hyphens':          rng.integers(0, 2, n_each),
        'num_at':               np.zeros(n_each, dtype=int),
        'num_question_marks':   rng.integers(0, 2, n_each),
        'num_equal':            rng.integers(0, 2, n_each),
        'num_slashes':          rng.integers(2, 6, n_each),
        'num_percent':          np.zeros(n_each, dtype=int),
        'has_https':            np.ones(n_each, dtype=int),
        'has_ip':               np.zeros(n_each, dtype=int),
        'has_suspicious_tld':   np.zeros(n_each, dtype=int),
        'num_subdomains':       rng.integers(0, 2, n_each),
        'has_phishing_keyword': np.zeros(n_each, dtype=int),
        'double_slash_in_path': np.zeros(n_each, dtype=int),
        'label':                np.ones(n_each, dtype=int),
    }
    df = pd.concat(
        [pd.DataFrame(phish), pd.DataFrame(legit)], ignore_index=True
    ).sample(frac=1, random_state=RANDOM_STATE)
    return df, 'Synthetic dataset (URL pattern statistics from phishing research)'

# ---- Auto-detect and load ----
arff_path = os.path.join('dataset', 'phishing.arff')
csv_path  = os.path.join('dataset', 'phishing.csv')

if os.path.exists(arff_path):
    df_raw, DATA_SOURCE = load_uci_arff(arff_path)
    print(f'[UCI ARFF] Loaded {len(df_raw):,} rows')
elif os.path.exists(csv_path):
    df_raw = pd.read_csv(csv_path)
    df_raw['label'] = df_raw[df_raw.columns[-1]].apply(lambda x: 0 if int(x)==-1 else 1)
    DATA_SOURCE = 'CSV from dataset/phishing.csv'
    print(f'[CSV] Loaded {len(df_raw):,} rows')
else:
    print('[INFO] No external dataset found. Using synthetic dataset.')
    print('       Download UCI dataset → dataset/phishing.arff for real data.')
    df_raw, DATA_SOURCE = build_synthetic_dataset(6000)

print(f'Dataset source : {DATA_SOURCE}')
print(f'Shape          : {df_raw.shape}')
df_raw.head()

## 4. Exploratory Data Analysis

In [ ]:
print('=== Basic Info ===')
print(f'Rows       : {len(df_raw):,}')
print(f'Columns    : {df_raw.shape[1]}')
print(f'Null values: {df_raw.isnull().sum().sum()}')
print()
print('=== Class Distribution ===')
label_counts = df_raw['label'].value_counts()
print(f'Legitimate (1): {label_counts.get(1, 0):,}')
print(f'Phishing   (0): {label_counts.get(0, 0):,}')

In [ ]:
# Class distribution pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = df_raw['label'].value_counts()
axes[0].pie(
    counts.values,
    labels=['Phishing (0)', 'Legitimate (1)'],
    colors=['#e74c3c', '#27ae60'],
    autopct='%1.1f%%',
    startangle=140,
    explode=(0.05, 0)
)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')

# Bar chart
axes[1].bar(
    ['Phishing', 'Legitimate'],
    counts.values,
    color=['#e74c3c', '#27ae60'],
    edgecolor='black'
)
axes[1].set_title('Class Count', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 20, str(v), ha='center', fontweight='bold')

plt.suptitle('Phishing vs Legitimate URL Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Statistical summary
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
feature_cols_eda = [c for c in numeric_cols if c != 'label']
df_raw[feature_cols_eda].describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# Correlation heatmap (top 16 numeric features)
sample_cols = feature_cols_eda[:16]
corr_df = df_raw[sample_cols + ['label']].copy().apply(pd.to_numeric, errors='coerce').fillna(0)
corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    linewidths=0.4, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Feature Engineering

We use the 16 URL-based features defined in `feature_extraction.py`.
When working with the UCI ARFF dataset we use the dataset's own numeric columns;
for the synthetic dataset the features are already in the correct form.

In [ ]:
# Feature demonstration on sample URLs
sys.path.insert(0, '.')
try:
    from feature_extraction import extract_features, FEATURE_NAMES
    sample_urls = [
        ('https://www.google.com',                              'Legitimate'),
        ('https://www.amazon.com/dp/B08N5KWB9H',               'Legitimate'),
        ('http://paypal-secure-login.tk/account/verify?u=123', 'Phishing'),
        ('http://192.168.1.1/update',                          'Phishing'),
        ('http://free-prize-winner.xyz/claim?id=abc123',       'Phishing'),
    ]
    rows = []
    for url, label in sample_urls:
        feat = extract_features(url)
        feat['url'] = url
        feat['true_label'] = label
        rows.append(feat)
    demo_df = pd.DataFrame(rows).set_index('url')
    print('Feature extraction demo:')
    demo_df
except ImportError:
    print('[WARN] feature_extraction.py not found in current directory.')
    print('       Run this notebook from the Phishing-Website-Detection/ folder.')

In [ ]:
# Prepare feature matrix X and target y
label_col    = 'label'
feature_cols = [c for c in df_raw.columns if c != label_col]

X = df_raw[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(float)
y = df_raw[label_col].values.astype(int)

print(f'Feature matrix shape : {X.shape}')
print(f'Target vector shape  : {y.shape}')
print(f'Class balance — Phishing: {(y==0).sum()}, Legitimate: {(y==1).sum()}')

## 6. Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set : {X_train.shape[0]:,} samples')
print(f'Test set     : {X_test.shape[0]:,} samples')
print(f'Train class balance — Phishing: {(y_train==0).sum()}, Legit: {(y_train==1).sum()}')
print(f'Test  class balance — Phishing: {(y_test==0).sum()},  Legit: {(y_test==1).sum()}')

## 7. Model Training

We train three classifiers and compare their performance:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    'Decision Tree': Pipeline([
        ('clf', DecisionTreeClassifier(max_depth=10, min_samples_split=5,
                                       random_state=RANDOM_STATE)),
    ]),
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=15,
                                       min_samples_split=5, n_jobs=-1,
                                       random_state=RANDOM_STATE)),
    ]),
}
print(f'Models to train: {list(models.keys())}')

In [ ]:
results = {}

for name, model in models.items():
    print(f'Training {name}...')
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    cv_scores = cross_val_score(
        model, X_train, y_train, cv=StratifiedKFold(5), scoring='f1'
    )

    results[name] = {
        'model':     model,
        'y_pred':    y_pred,
        'y_proba':   y_proba,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred, zero_division=0),
        'f1':        f1_score(y_test, y_pred, zero_division=0),
        'auc':       roc_auc_score(y_test, y_proba) if y_proba is not None else None,
        'cv_mean':   cv_scores.mean(),
        'cv_std':    cv_scores.std(),
        'time':      elapsed,
    }
    print(f'  Accuracy={results[name]["accuracy"]:.4f}  F1={results[name]["f1"]:.4f}  '
          f'AUC={results[name]["auc"]:.4f}  ({elapsed:.1f}s)')

print('\nAll models trained ✅')

## 8. Model Evaluation

In [ ]:
# Summary comparison table
summary_rows = []
for name, res in results.items():
    summary_rows.append({
        'Model':       name,
        'Accuracy':    f"{res['accuracy']:.4f}",
        'Precision':   f"{res['precision']:.4f}",
        'Recall':      f"{res['recall']:.4f}",
        'F1-Score':    f"{res['f1']:.4f}",
        'AUC-ROC':     f"{res['auc']:.4f}" if res['auc'] else 'N/A',
        'CV F1 (mean±std)': f"{res['cv_mean']:.4f} ± {res['cv_std']:.4f}",
        'Train Time':  f"{res['time']:.2f}s",
    })
pd.DataFrame(summary_rows).set_index('Model')

In [ ]:
# Grouped bar chart — model comparison
metrics  = ['accuracy', 'precision', 'recall', 'f1']
names    = list(results.keys())
x        = np.arange(len(names))
width    = 0.2
colors   = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

fig, ax = plt.subplots(figsize=(10, 5))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = [results[n][metric] for n in names]
    ax.bar(x + i * width, vals, width, label=metric.capitalize(), color=color, alpha=0.85)

ax.set_xticks(x + width * (len(metrics) - 1) / 2)
ax.set_xticklabels(names, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
plt.tight_layout()
os.makedirs('reports', exist_ok=True)
plt.savefig('reports/model_comparison.png', dpi=120)
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Phishing', 'Legit'],
        yticklabels=['Phishing', 'Legit'],
        ax=ax
    )
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/confusion_matrix.png', dpi=120)
plt.show()

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(7, 5))
for name, res in results.items():
    if res['y_proba'] is not None:
        fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
        ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", linewidth=2)
ax.plot([0,1], [0,1], 'k--', label='Random Classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('reports/roc_curve.png', dpi=120)
plt.show()

In [ ]:
# Full classification reports
for name, res in results.items():
    print(f'\n{"="*50}')
    print(f'  {name}')
    print(f'{"="*50}')
    print(classification_report(
        y_test, res['y_pred'],
        target_names=['Phishing', 'Legitimate'],
        zero_division=0
    ))

## 9. Best Model Selection & Saving

In [ ]:
# Select best model by F1-score
best_name = max(results, key=lambda k: results[k]['f1'])
best      = results[best_name]

print(f'Best model : {best_name}')
print(f'Accuracy   : {best["accuracy"]:.4f}')
print(f'F1-Score   : {best["f1"]:.4f}')
print(f'AUC-ROC    : {best["auc"]:.4f}')

In [ ]:
# Feature importance (Random Forest / Decision Tree)
clf = best['model'][-1] if hasattr(best['model'], '__len__') else best['model']
if hasattr(clf, 'feature_importances_'):
    importances = clf.feature_importances_
    idx = np.argsort(importances)[::-1]

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#c0392b' if i < 5 else '#2980b9' for i in range(len(importances))]
    ax.bar(range(len(importances)), importances[idx], color=colors)
    ax.set_xticks(range(len(importances)))
    ax.set_xticklabels(
        [feature_cols[i] for i in idx], rotation=45, ha='right', fontsize=8
    )
    ax.set_title(f'Feature Importances — {best_name}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Importance')
    plt.tight_layout()
    plt.savefig('reports/feature_importance.png', dpi=120)
    plt.show()
elif hasattr(clf, 'coef_'):
    importances = np.abs(clf.coef_[0])
    idx = np.argsort(importances)[::-1]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(len(importances)), importances[idx], color='#2980b9')
    ax.set_xticks(range(len(importances)))
    ax.set_xticklabels(
        [feature_cols[i] for i in idx], rotation=45, ha='right', fontsize=8
    )
    ax.set_title(f'Coefficient Magnitudes — {best_name}', fontsize=13, fontweight='bold')
    ax.set_ylabel('|Coefficient|')
    plt.tight_layout()
    plt.savefig('reports/feature_importance.png', dpi=120)
    plt.show()

In [ ]:
# Save the best model
os.makedirs('models', exist_ok=True)

metadata_dict = {
    'model_name':     best_name,
    'feature_names':  feature_cols,
    'accuracy':       best['accuracy'],
    'precision':      best['precision'],
    'recall':         best['recall'],
    'f1':             best['f1'],
    'auc':            best['auc'],
    'dataset_source': DATA_SOURCE,
    'n_train':        len(X_train),
    'n_test':         len(X_test),
}

bundle = {
    'model':         best['model'],
    'metadata':      metadata_dict,
    'feature_names': feature_cols,
}

joblib.dump(bundle, 'models/model.pkl')
joblib.dump(bundle, 'model.pkl')  # project root convenience copy

with open('models/model_metadata.json', 'w') as f:
    json.dump({k: v for k, v in metadata_dict.items()
               if not isinstance(v, np.ndarray)}, f, indent=2)

print(f'Model saved to models/model.pkl  ✅')
print(f'Model saved to model.pkl         ✅')
print(f'Metadata saved to models/model_metadata.json  ✅')

## 10. Live URL Prediction Demo

In [ ]:
def predict_url(url: str, model_bundle: dict) -> dict:
    """
    Predict whether a URL is phishing or legitimate.

    Returns a dict with:
        prediction  : 'Phishing' or 'Legitimate'
        confidence  : float 0–1
        features    : dict of extracted features
    """
    try:
        from feature_extraction import extract_features
        feat_dict = extract_features(url)
        f_names   = model_bundle['feature_names']
        vec = [feat_dict.get(f, 0) for f in f_names]
    except ImportError:
        # Notebook running without feature_extraction module
        # fall back to zero vector (demo only)
        f_names = feature_cols
        vec     = [0] * len(f_names)
        feat_dict = {}

    X_input = np.array([vec])
    pred = model_bundle['model'].predict(X_input)[0]
    conf = None
    if hasattr(model_bundle['model'], 'predict_proba'):
        proba = model_bundle['model'].predict_proba(X_input)[0]
        conf  = proba[1] if pred == 1 else proba[0]

    return {
        'prediction': 'Legitimate ✅' if pred == 1 else 'Phishing 🚨',
        'confidence': conf,
        'features':   feat_dict,
    }


# Load saved model and test
saved_bundle = joblib.load('model.pkl')

demo_urls = [
    'https://www.google.com',
    'https://www.amazon.com/dp/B08N5KWB9H',
    'http://paypal-secure-login.tk/account/verify?user=abc',
    'http://192.168.1.1/update-credentials',
    'https://github.com/user/repo',
    'http://free-prize-winner.xyz/claim?id=12345',
    'http://microsoft-support-alert.cc/fix-now',
]

print(f'{"URL":<60} {"Prediction":<20} {"Confidence"}')
print('-' * 95)
for url in demo_urls:
    result = predict_url(url, saved_bundle)
    conf_str = f"{result['confidence']:.1%}" if result['confidence'] else 'N/A'
    print(f"{url:<60} {result['prediction']:<20} {conf_str}")

## 11. Summary

### Results

| Model | Accuracy | F1-Score | AUC-ROC |
|---|---|---|---|
| Logistic Regression | (see output above) | | |
| Decision Tree | | | |
| Random Forest | | | |

> **Run the cells above to populate the actual values.**

### Conclusions

1. **Random Forest** typically achieves the highest accuracy and F1-score on URL-based
   phishing detection tasks due to its ensemble nature and ability to capture non-linear
   feature interactions.

2. The most discriminating features are:
   - `has_https` — HTTPS usage
   - `has_suspicious_tld` — free/abused TLDs
   - `has_phishing_keyword` — common phishing terms
   - `url_length` — phishing URLs tend to be longer
   - `num_subdomains` — multiple subdomain levels

3. **Limitations:**
   - The system analyses URL structure only — no live page content is inspected.
   - A URL that looks legitimate can still host phishing content.
   - The confidence score reflects model uncertainty, not guaranteed website safety.

### How to Run the Streamlit App
```bash
cd Phishing-Website-Detection
pip install -r requirements.txt
python train_model.py          # train model once
streamlit run app.py           # launch dashboard
```

---
*IBM SkillsBuild Cyber Security BootCamp 2026 — BharatCares*  
*Developed by Rampravesh*